In [1]:
NAME = "a2-640-kaggle"
IMGSZ = 640
EPOCHS = 50
FRACTION = 1.0

In [2]:
!git clone https://github.com/mattbuildz/shelf-liquor-detection.git /kaggle/working/ML
%cd /kaggle/working/ML
!pip install -q ultralytics==8.4.51 pycocotools
!mkdir -p data/train
!cp -r /kaggle/input/datasets/mateuszglab/shelf-liquor-train/train/images data/train/images
!cp /kaggle/input/datasets/mateuszglab/shelf-liquor-train/train/annotations.json data/train/

Cloning into '/kaggle/working/ML'...
remote: Enumerating objects: 121, done.
remote: Counting objects: 100% (121/121), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 121 (delta 42), reused 101 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (121/121), 4.40 MiB | 40.95 MiB/s, done.
Resolving deltas: 100% (42/42), done.
/kaggle/working/ML
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 44.0 MB/s eta 0:00:00


In [3]:
import os
!python scripts/training/convert_coco_to_yolo.py
!python scripts/training/make_dataset_yaml.py
n_img = len(os.listdir("data/train/images"))
n_lab = len(os.listdir("data/train/labels"))
print(n_img, "images,", n_lab, "labels")
assert n_img == 3904
assert n_lab == 1123

3904 images, 1123 labels


In [4]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")
results = model.train(data="scripts/training/dataset.yaml", epochs=EPOCHS, fraction=FRACTION,
                      imgsz=IMGSZ, batch=16, device=0, optimizer="AdamW", lr0=0.001,
                      cache="ram", name=NAME)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
New https://pypi.org/project/ultralytics/8.4.174 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.51 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=scripts/training/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0

In [5]:
!rm -rf data
!ls runs/detect/{NAME}/weights

best.pt  last.pt
